<a href="https://colab.research.google.com/github/Swastika200105/My-daily-data-science-journal/blob/main/Data_analysis_of_student_attendance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart,LineChart, PieChart, Reference
from openpyxl.utils import get_column_letter

df = pd.read_excel("/content/Student_Attendance_Big_Dataset.xlsx")
df["Date"] = pd.to_datetime(df["Date"])# convert Date column  to actual datetime
#create present_flag
df["Present_Flag"] = (df["Status"].str.strip().str.lower() == "present").astype(int)
# create absent_flag
df["Absent_Flag"] = (df["Status"].str.strip().str.lower() == "absent").astype(int)


# Core KPI calculations
total_records = len(df)#Total number of attendance records
students =  df["Student_ID"].nunique() #Number of unique students
dates = df["Date"].nunique() #Number of unique dates
present = df["Present_Flag"].sum() #Total number of present records
absent = df["Absent_Flag"].sum() #Total number of absent records
attendance_rate = (present / total_records) * 100 #attendance rate calculation

# Summary tables

class_summary = df.groupby("Class").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
class_summary["Attendance_Rate_%"] = class_summary["Present"] / class_summary["total_records"] * 100
class_summary = class_summary.sort_values(by="Attendance_Rate_%", ascending=False)

section_summary = df.groupby("Section").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
section_summary["Attendance_Rate_%"] = section_summary["Present"] / section_summary["total_records"] * 100
section_summary = section_summary.sort_values(by="Attendance_Rate_%", ascending=False)

monthly = df.assign(Month=df["Date"].dt.to_period("M")).groupby("Month").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
monthly["Attendance_Rate_%"] = monthly["Present"] / monthly["total_records"] * 100
daily = df.groupby("Date").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
daily["Attendance_Rate_%"] = daily["Present"] / daily["total_records"] * 100
daily = daily.sort_values("Date")

student_summary = df.groupby(["Student_ID", "Student_Name"]).agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
student_summary["Attendance_Rate_%"] = student_summary["Present"] / student_summary["total_records"] * 100
student_summary["Risk_Level"] = np.select (
  [
       student_summary["Attendance_Rate_%"] < 75,
       student_summary["Attendance_Rate_%"] < 85,
       student_summary["Attendance_Rate_%"] < 90
  ],
  ["Critical", "High", "Watch"],
  default = "Good"
)

student_summary = student_summary.sort_values(["Attendance_Rate_%", "Absent"], ascending=[True, False])


class_section = df.groupby(["Class", "Section"]).agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
class_section["Attendance_Rate_%"] = class_section["Present"] / class_section["total_records"] * 100
class_section = class_section.sort_values("Attendance_Rate_%", ascending= False)


weekday = df.assign(Weekday=df["Date"].dt.day_name()).groupby("Weekday").agg(
    total_records = ("Status", "size"),
    Present = ("Present_Flag", "sum"),
    Absent = ("Absent_Flag", "sum")
).reset_index()
weekday["Attendance_Rate_%"] = weekday["Present"] / weekday["total_records"] * 100
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday["Weekday"] = pd.Categorical(weekday["Weekday"], categories=weekday_order, ordered=True)
weekday = weekday.sort_values("Weekday")
weekday["Weekday"] = weekday["Weekday"].astype(str)


In [22]:
df.head()

,Student_ID,Student_Name,Class,Section,Date,Status,Present_Flag,Absent_Flag
0,1,Student_1,Grade 10,B,2026-01-01,Present,1,0
1,2,Student_2,Grade 9,C,2026-01-01,Present,1,0
2,3,Student_3,Grade 9,B,2026-01-01,Present,1,0
3,4,Student_4,Grade 7,A,2026-01-01,Present,1,0
4,5,Student_5,Grade 7,B,2026-01-01,Present,1,0
